# Script 1 [EfficientNet-B3]

In [1]:
"""
S1_00_config.txt
Script 1 — EfficientNet-B3 (Image-Only Ablation)
Shared configuration — paths, class mapping, hyperparameters, checkpoint settings.
Every other Script 1 file imports from this one.
"""

from pathlib import Path

# --- Base paths ---
BASE_DIR = Path(r"T:\Thesis - Development")
FOLD_ASSIGNMENTS_PATH = BASE_DIR / "Resources" / "Artifacts" / "Phase 1 - P2" / "cv_fold_assignments.csv"
IMAGES_DIR = BASE_DIR / "Resources" / "Artifacts" / "Phase 1 - P2" / "Final Output Phase 1" / "images"

# --- Output paths (this script — EfficientNet-B3) ---
OUTPUT_DIR = BASE_DIR / "Resources" / "Artifacts" / "Phase 2 - P1" / "EfficientNet-B3"
CHECKPOINTS_DIR = OUTPUT_DIR / "checkpoints"
BEST_WEIGHTS_DIR = OUTPUT_DIR / "best_weights"
RESULTS_DIR = OUTPUT_DIR / "results"

for d in (CHECKPOINTS_DIR, BEST_WEIGHTS_DIR, RESULTS_DIR):
    d.mkdir(parents=True, exist_ok=True)

# --- Fixed class index mapping (shared across ALL Phase 2 scripts + later phases) ---
CLASS_TO_IDX = {
    "akiec": 0,
    "bcc": 1,
    "bkl": 2,
    "df": 3,
    "mel": 4,
    "nv": 5,
    "vasc": 6,
}
IDX_TO_CLASS = {v: k for k, v in CLASS_TO_IDX.items()}
NUM_CLASSES = len(CLASS_TO_IDX)

# --- Reproducibility ---
RANDOM_SEED = 42  # matches Phase 1 P2's locked seed

# --- Image ---
IMAGE_SIZE = 300

# --- Class-balanced loss (locked, Phase 1 P2 Section 10) ---
BETA = 0.999

# --- Optimizer / training (locked, Phase 2_Implementation.md) ---
WEIGHT_DECAY = 1e-5
LEARNING_RATE = 1e-4  # flat, whole model — no freeze phase for baselines

# --- Early stopping ---
EARLY_STOP_PATIENCE = 10
MAX_EPOCHS = 100  # raised from 50 -- some runs may still be improving near the old cap
EARLY_STOP_METRIC = "val_auc_roc"

# --- Checkpointing ---
CHECKPOINT_EVERY_N_EPOCHS = 1     # every epoch
CHECKPOINT_KEEP_HISTORY = False   # rolling only — latest overwrites previous


def checkpoint_path(fold_num: int) -> Path:
    """Rolling checkpoint for a given fold — overwritten each epoch. Crash-recovery only."""
    return CHECKPOINTS_DIR / f"fold{fold_num}_latest.pt"


def best_weights_path(fold_num: int) -> Path:
    """Final best-epoch weights for a given fold — the artifact Phase 6 uses."""
    return BEST_WEIGHTS_DIR / f"fold{fold_num}_best.pt"


print("S1_00_config loaded.")
print(f"  Fold assignments: {FOLD_ASSIGNMENTS_PATH}")
print(f"  Images dir:       {IMAGES_DIR}")
print(f"  Output dir:       {OUTPUT_DIR}")

S1_00_config loaded.
  Fold assignments: T:\Thesis - Development\Resources\Artifacts\Phase 1 - P2\cv_fold_assignments.csv
  Images dir:       T:\Thesis - Development\Resources\Artifacts\Phase 1 - P2\Final Output Phase 1\images
  Output dir:       T:\Thesis - Development\Resources\Artifacts\Phase 2 - P1\EfficientNet-B3


In [2]:
"""
S1_01_preprocessing.txt
Script 1 — EfficientNet-B3 (Image-Only Ablation)
Image loading + ImageNet normalization. Reusable across train and validation —
no augmentation here (that is conditional on split, handled in S1_02_dataset).
"""

import cv2
import numpy as np
import torch

# Requires S1_00_config already run in this notebook (defines IMAGES_DIR).

# Standard ImageNet normalization constants (Phase 1 P2, Section 3.2)
IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)


def load_image(image_id: str) -> np.ndarray:
    """
    Load a processed image by image_id from IMAGES_DIR.
    Returns an RGB uint8 array (H, W, 3). Images are already 300x300 from
    Phase 1 P2's Section 12 pipeline -- no resize applied here.
    """
    img_path = IMAGES_DIR / f"{image_id}.jpg"
    image_bgr = cv2.imread(str(img_path))
    if image_bgr is None:
        raise FileNotFoundError(f"Could not load image: {img_path}")
    return cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)


def normalize_image(image_rgb: np.ndarray) -> torch.Tensor:
    """
    Scale to [0, 1] and apply ImageNet per-channel mean/std normalization.
    Returns a torch.Tensor in C x H x W format.
    """
    image_float = image_rgb.astype(np.float32) / 255.0
    normalized = (image_float - IMAGENET_MEAN) / IMAGENET_STD
    tensor = torch.from_numpy(normalized).permute(2, 0, 1).float()
    return tensor

In [3]:
"""
augment_image_carryover.txt
Carried over unchanged from Phase 1 P2's 08_07_augmentation.txt.
Run this cell before S1_02_dataset -- Phase 2-P1 is a new notebook and
does not have this function defined yet.
"""

import cv2
import numpy as np

AUGMENTATION_CONFIG = {
    "hflip_prob": 0.5,
    "vflip_prob": 0.5,
    "rotation_max_degrees": 180,
    "zoom_range": (0.9, 1.1),
    "brightness_range": (0.8, 1.2),
}


def augment_image(image, config=AUGMENTATION_CONFIG):
    """
    image: square image (BGR or RGB -- purely geometric/intensity operations),
           typically the resized 300x300 output.
    Returns: augmented image, same shape as input.

    Applies a random draw of each augmentation, independently, per call.
    Intended to be called once per image, per epoch, inside the training
    DataLoader -- never pre-applied and saved to disk.
    """
    h, w = image.shape[:2]
    out = image.copy()

    # Horizontal flip
    if np.random.random() < config["hflip_prob"]:
        out = cv2.flip(out, 1)

    # Vertical flip
    if np.random.random() < config["vflip_prob"]:
        out = cv2.flip(out, 0)

    # Rotation (up to +/- rotation_max_degrees)
    angle = np.random.uniform(-config["rotation_max_degrees"], config["rotation_max_degrees"])
    center = (w / 2, h / 2)
    rotation_matrix = cv2.getRotationMatrix2D(center, angle, 1.0)
    out = cv2.warpAffine(out, rotation_matrix, (w, h), borderMode=cv2.BORDER_REFLECT_101)

    # Zoom (+/- range) -- scale then centre-crop or pad back to original size
    zoom_factor = np.random.uniform(*config["zoom_range"])
    new_h, new_w = int(round(h * zoom_factor)), int(round(w * zoom_factor))
    scaled = cv2.resize(out, (new_w, new_h), interpolation=cv2.INTER_LINEAR)

    if zoom_factor >= 1.0:
        # Zoomed in -- centre-crop back down to original size
        row_start = (new_h - h) // 2
        col_start = (new_w - w) // 2
        out = scaled[row_start:row_start + h, col_start:col_start + w]
    else:
        # Zoomed out -- pad back up to original size
        pad_h = h - new_h
        pad_w = w - new_w
        top, bottom = pad_h // 2, pad_h - pad_h // 2
        left, right = pad_w // 2, pad_w - pad_w // 2
        out = cv2.copyMakeBorder(scaled, top, bottom, left, right, borderType=cv2.BORDER_REFLECT_101)

    # Brightness
    brightness_factor = np.random.uniform(*config["brightness_range"])
    out = np.clip(out.astype(np.float32) * brightness_factor, 0, 255).astype(np.uint8)

    return out

In [4]:
"""
S1_02_dataset.txt
Script 1 — EfficientNet-B3 (Image-Only Ablation)
PyTorch Dataset class + fold dataframe helper.

Requires already run in this notebook:
  - S1_00_config      (FOLD_ASSIGNMENTS_PATH, CLASS_TO_IDX)
  - S1_01_preprocessing (load_image, normalize_image)
  - augment_image()   -- reused from Phase 1 P2's 08_07_augmentation.txt;
                          must be pasted into this notebook before this cell.
"""

import pandas as pd
import torch
from torch.utils.data import Dataset


def get_fold_dataframes(fold_num: int):
    """
    Read cv_fold_assignments.csv and split into (train_df, val_df) for the
    given fold. Training = every row where cv_fold != fold_num (the other
    4 folds combined). Validation = cv_fold == fold_num.
    """
    fold_df = pd.read_csv(FOLD_ASSIGNMENTS_PATH)
    train_df = fold_df[fold_df["cv_fold"] != fold_num].reset_index(drop=True)
    val_df = fold_df[fold_df["cv_fold"] == fold_num].reset_index(drop=True)
    return train_df, val_df


class HAM10000Dataset(Dataset):
    """
    is_train=True  -> augmentation applied (training split)
    is_train=False -> no augmentation (validation split)
    """

    def __init__(self, fold_df: pd.DataFrame, is_train: bool):
        self.fold_df = fold_df.reset_index(drop=True)
        self.is_train = is_train

    def __len__(self):
        return len(self.fold_df)

    def __getitem__(self, idx):
        row = self.fold_df.iloc[idx]
        image_id = row["image_id"]
        label = CLASS_TO_IDX[row["dx"]]

        image_rgb = load_image(image_id)

        if self.is_train:
            image_rgb = augment_image(image_rgb)

        tensor = normalize_image(image_rgb)
        return tensor, torch.tensor(label, dtype=torch.long)

In [5]:
"""
S1_03_class_balancing.txt
Script 1 — EfficientNet-B3 (Image-Only Ablation)
Effective-number loss weights + balanced mini-batch sampler.
Two GENUINELY SEPARATE mechanisms (Section 3.4) -- a bug can silently
disable one while the other still looks correct, so they stay independent
here and must be verified independently once wired into the DataLoader.

Requires already run in this notebook:
  - S1_00_config (BETA, RANDOM_SEED, CLASS_TO_IDX, IDX_TO_CLASS, NUM_CLASSES)
"""

import numpy as np
import pandas as pd
import torch
from torch.utils.data import WeightedRandomSampler


def compute_effective_number_weights(class_counts, beta):
    """
    class_counts: dict {class_name: count} or pandas Series
    beta: float, locked at BETA (0.999)

    Returns: dict {class_name: weight}, normalised so weights sum to
             len(class_counts) (mean weight = 1.0)
    Unchanged from Phase 1 P2's 10_03.
    """
    classes = list(class_counts.keys()) if isinstance(class_counts, dict) else class_counts.index.tolist()
    counts = np.array([class_counts[c] for c in classes], dtype=np.float64)

    effective_num = (1.0 - np.power(beta, counts)) / (1.0 - beta)
    raw_weights = 1.0 / effective_num

    normalized_weights = raw_weights / raw_weights.sum() * len(classes)
    return dict(zip(classes, normalized_weights))


def get_fold_class_weights(train_df: pd.DataFrame, beta: float = BETA) -> torch.Tensor:
    """
    Computes effective-number loss weights from THIS FOLD'S OWN training
    counts (never global/validation/holdout counts -- Section 3.4's
    leakage warning). Returns a tensor ordered by CLASS_TO_IDX so it lines
    up directly with the model's output indices for CrossEntropyLoss.
    """
    class_counts = train_df["dx"].value_counts().to_dict()
    weights_by_class = compute_effective_number_weights(class_counts, beta)
    ordered = [weights_by_class[IDX_TO_CLASS[i]] for i in range(NUM_CLASSES)]
    return torch.tensor(ordered, dtype=torch.float32)


def build_balanced_sampler(labels, seed: int = RANDOM_SEED) -> WeightedRandomSampler:
    """
    labels: array-like of dx strings, one per training sample, in dataset order.
    seed: seeds the sampler's own torch.Generator (independent of global
          random state), so draw order is reproducible and resumable.

    Uses plain inverse class-frequency per sample (distinct from the
    effective-number formula above, which is loss-specific).
    Returns: WeightedRandomSampler, with replacement, num_samples=len(labels).
    The generator is reachable via sampler.generator for checkpointing.
    """
    labels_series = pd.Series(labels)
    class_counts = labels_series.value_counts()

    class_weight_map = 1.0 / class_counts
    sample_weights = labels_series.map(class_weight_map).values

    generator = torch.Generator()
    generator.manual_seed(seed)

    sampler = WeightedRandomSampler(
        weights=torch.as_tensor(sample_weights, dtype=torch.double),
        num_samples=len(labels_series),
        replacement=True,
        generator=generator,
    )
    return sampler

In [6]:
"""
S1_04_model.txt
Script 1 — EfficientNet-B3 (Image-Only Ablation)
Model architecture only. No metadata branch, no fusion mechanism, no
freeze/unfreeze phases (baselines train end-to-end from epoch 1).

Head order: Dropout(0.5) -> Linear(->7) -- conventional order, matches
torchvision's own EfficientNet head structure (confirmed, not the literal
"dense then dropout" reading of Section 4.4).

Model outputs RAW LOGITS, not softmax probabilities. CrossEntropyLoss
applies softmax internally during training; an explicit torch.softmax
call is applied separately, at inference/metrics time only (S1_07).
Outputting softmax from the model itself would double-softmax the loss
and produce incorrect gradients.

Requires already run in this notebook:
  - S1_00_config (NUM_CLASSES)
"""

import torch.nn as nn
from torchvision.models import efficientnet_b3, EfficientNet_B3_Weights


def build_model(num_classes: int = NUM_CLASSES) -> nn.Module:
    """
    EfficientNet-B3, pretrained on ImageNet, with its default 1000-class
    head replaced by a plain classification head for the 7-class task.
    """
    model = efficientnet_b3(weights=EfficientNet_B3_Weights.IMAGENET1K_V1)

    in_features = model.classifier[1].in_features  # 1536, per Section 4.2
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.5, inplace=True),
        nn.Linear(in_features, num_classes),
    )
    return model

In [7]:
"""
S1_05_checkpointing.txt
Script 1 — EfficientNet-B3 (Image-Only Ablation)
Save/load logic for resumable training. Two separate save points:

  - save_checkpoint / load_checkpoint: rolling, per-epoch crash-recovery
    state (checkpoints\). Overwritten each epoch. Restores model,
    optimizer, epoch number, early-stopping state, and the balanced
    sampler's generator state (deterministic continuation, not a fresh
    random restart, per agreed design).

  - save_best_weights: separate, smaller save (best_weights\) — the
    final per-fold artifact Phase 6 uses. Triggered only on a new best
    validation AUC-ROC, untouched by ordinary epoch checkpointing.

Requires already run in this notebook:
  - S1_00_config (checkpoint_path, best_weights_path)
"""

import torch


def save_checkpoint(fold_num, epoch, model, optimizer, sampler, best_val_auc, patience_counter):
    """
    Overwrites the rolling checkpoint for this fold. Call at the end of
    every completed epoch.
    """
    checkpoint = {
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "sampler_generator_state": sampler.generator.get_state(),
        "best_val_auc": best_val_auc,
        "patience_counter": patience_counter,
    }
    torch.save(checkpoint, checkpoint_path(fold_num))


def load_checkpoint(fold_num, model, optimizer, sampler, device="cpu"):
    """
    If a checkpoint exists for this fold: restores model, optimizer, and
    sampler generator state IN PLACE, and returns
      {"epoch": ..., "best_val_auc": ..., "patience_counter": ...}
    so the training loop knows where to resume from.

    If no checkpoint exists yet for this fold: returns None (fresh start).
    """
    path = checkpoint_path(fold_num)
    if not path.exists():
        return None

    checkpoint = torch.load(path, map_location=device)
    model.load_state_dict(checkpoint["model_state_dict"])
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    sampler.generator.set_state(checkpoint["sampler_generator_state"])

    return {
        "epoch": checkpoint["epoch"],
        "best_val_auc": checkpoint["best_val_auc"],
        "patience_counter": checkpoint["patience_counter"],
    }


def save_best_weights(fold_num, model):
    """Saves model weights only -- the final artifact for this fold."""
    torch.save(model.state_dict(), best_weights_path(fold_num))

In [8]:
"""
S1_06_train_loop.txt
Script 1 — EfficientNet-B3 (Image-Only Ablation)
Per-fold training loop. Resumable at the epoch level via S1_05's
checkpointing. Computes a lightweight macro one-vs-rest AUC-ROC each
epoch for early-stopping/best-checkpoint decisions only -- the full
per-class metric suite (F1, sensitivity, specificity, accuracy) is
computed once after training, in S1_07, from the saved best weights.

Requires already run in this notebook:
  - S1_00_config, S1_01_preprocessing, S1_02_dataset,
    S1_03_class_balancing, S1_04_model, S1_05_checkpointing,
    augment_image_carryover
"""

import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.utils.data import DataLoader
from sklearn.metrics import roc_auc_score
from tqdm import tqdm

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 16  # starting value -- adjust after a first run shows GPU memory usage


def train_fold(fold_num: int) -> dict:
    """
    Trains EfficientNet-B3 (image-only ablation) on the given fold,
    resuming from a checkpoint if one exists. Returns a summary dict
    with the fold's final best_val_auc and how training ended.
    """
    train_df, val_df = get_fold_dataframes(fold_num)

    train_dataset = HAM10000Dataset(train_df, is_train=True)
    val_dataset = HAM10000Dataset(val_df, is_train=False)

    sampler = build_balanced_sampler(train_df["dx"].values, seed=RANDOM_SEED)
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler, num_workers=0)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

    model = build_model().to(DEVICE)
    optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

    class_weights = get_fold_class_weights(train_df).to(DEVICE)
    criterion = nn.CrossEntropyLoss(weight=class_weights)

    # --- Resume if a checkpoint exists for this fold ---
    resumed = load_checkpoint(fold_num, model, optimizer, sampler, device=DEVICE)
    if resumed is not None:
        start_epoch = resumed["epoch"] + 1
        best_val_auc = resumed["best_val_auc"]
        patience_counter = resumed["patience_counter"]
        print(f"Fold {fold_num}: resuming from epoch {start_epoch} "
              f"(best_val_auc so far: {best_val_auc:.4f})")
    else:
        start_epoch = 0
        best_val_auc = -float("inf")
        patience_counter = 0
        print(f"Fold {fold_num}: starting fresh")

    early_stopped = False
    final_epoch = start_epoch - 1

    for epoch in range(start_epoch, MAX_EPOCHS):
        # --- Train pass ---
        model.train()
        running_loss = 0.0
        for images, labels in tqdm(train_loader, desc=f"Fold {fold_num} | Epoch {epoch} | train", leave=False):
            images, labels = images.to(DEVICE), labels.to(DEVICE)

            optimizer.zero_grad()
            logits = model(images)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * images.size(0)

        train_loss = running_loss / len(train_dataset)

        # --- Validation pass ---
        model.eval()
        all_labels, all_probs = [], []
        with torch.no_grad():
            for images, labels in val_loader:
                images = images.to(DEVICE)
                logits = model(images)
                probs = torch.softmax(logits, dim=1)
                all_labels.append(labels.numpy())
                all_probs.append(probs.cpu().numpy())

        all_labels = np_concat(all_labels)
        all_probs = np_concat(all_probs)
        val_auc = roc_auc_score(all_labels, all_probs, multi_class="ovr", average="macro")

        # --- Best-checkpoint / early-stopping update ---
        if val_auc > best_val_auc:
            best_val_auc = val_auc
            patience_counter = 0
            save_best_weights(fold_num, model)
        else:
            patience_counter += 1

        final_epoch = epoch
        save_checkpoint(fold_num, epoch, model, optimizer, sampler, best_val_auc, patience_counter)

        print(f"Fold {fold_num} | Epoch {epoch} | train_loss={train_loss:.4f} "
              f"| val_auc={val_auc:.4f} | best={best_val_auc:.4f} | patience={patience_counter}")

        if patience_counter >= EARLY_STOP_PATIENCE:
            early_stopped = True
            print(f"Fold {fold_num}: early stopping at epoch {epoch}")
            break

    summary = {
        "fold": fold_num,
        "best_val_auc": best_val_auc,
        "final_epoch": final_epoch,
        "early_stopped": early_stopped,
    }

    # Completion marker -- distinct from best_weights, which gets saved
    # mid-training too. S1_08 checks for THIS file to know a fold is done.
    import json
    with open(RESULTS_DIR / f"fold{fold_num}_training_summary.json", "w") as f:
        json.dump(summary, f)

    return summary


def np_concat(list_of_arrays):
    """Small helper -- avoids importing numpy just for np.concatenate at call sites."""
    import numpy as np
    return np.concatenate(list_of_arrays, axis=0)

In [9]:
"""
S1_07_metrics.txt
Script 1 — EfficientNet-B3 (Image-Only Ablation)
Post-training evaluation. Runs once per fold, using that fold's saved
best weights. Computes BOTH macro and per-class metrics; sensitivity
and specificity are one-vs-rest per class.

Saves two files per fold to RESULTS_DIR:
  - fold{n}_val_predictions.csv  (image_id, true_label, per-class probs)
    -- raw predictions, so Phase 6's DeLong tests don't need to re-run
    inference from scratch.
  - fold{n}_per_class_metrics.csv (class, auc_roc, f1, sensitivity, specificity)

Returns macro metrics + accuracy for S1_08 to aggregate across folds.

Requires already run in this notebook:
  - S1_00_config, S1_02_dataset, S1_04_model, S1_06_train_loop (DEVICE, BATCH_SIZE)
"""

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score, confusion_matrix


def evaluate_fold(fold_num: int) -> dict:
    _, val_df = get_fold_dataframes(fold_num)
    val_dataset = HAM10000Dataset(val_df, is_train=False)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

    model = build_model().to(DEVICE)
    model.load_state_dict(torch.load(best_weights_path(fold_num), map_location=DEVICE))
    model.eval()

    all_labels, all_probs = [], []
    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(DEVICE)
            logits = model(images)
            probs = torch.softmax(logits, dim=1)
            all_labels.append(labels.numpy())
            all_probs.append(probs.cpu().numpy())

    all_labels = np.concatenate(all_labels, axis=0)
    all_probs = np.concatenate(all_probs, axis=0)
    y_pred = all_probs.argmax(axis=1)

    # --- Save raw predictions (image_id, true_label, per-class probs) ---
    pred_df = pd.DataFrame(all_probs, columns=[IDX_TO_CLASS[i] for i in range(NUM_CLASSES)])
    pred_df.insert(0, "true_label", [IDX_TO_CLASS[i] for i in all_labels])
    pred_df.insert(0, "image_id", val_df["image_id"].tolist())
    pred_df.to_csv(RESULTS_DIR / f"fold{fold_num}_val_predictions.csv", index=False)

    # --- Per-class metrics (all one-vs-rest) ---
    cm = confusion_matrix(all_labels, y_pred, labels=range(NUM_CLASSES))
    f1_per_class = f1_score(all_labels, y_pred, labels=list(range(NUM_CLASSES)), average=None, zero_division=0)

    per_class_rows = []
    for i in range(NUM_CLASSES):
        y_true_binary = (all_labels == i).astype(int)
        class_auc = roc_auc_score(y_true_binary, all_probs[:, i])

        tp = cm[i, i]
        fn = cm[i, :].sum() - tp
        fp = cm[:, i].sum() - tp
        tn = cm.sum() - tp - fn - fp

        sensitivity = tp / (tp + fn) if (tp + fn) > 0 else float("nan")
        specificity = tn / (tn + fp) if (tn + fp) > 0 else float("nan")

        per_class_rows.append({
            "class": IDX_TO_CLASS[i],
            "auc_roc": class_auc,
            "f1": f1_per_class[i],
            "sensitivity": sensitivity,
            "specificity": specificity,
        })

    per_class_df = pd.DataFrame(per_class_rows)
    per_class_df.to_csv(RESULTS_DIR / f"fold{fold_num}_per_class_metrics.csv", index=False)

    # --- Macro / aggregate metrics ---
    macro_metrics = {
        "fold": fold_num,
        "macro_auc_roc": per_class_df["auc_roc"].mean(),
        "macro_f1": per_class_df["f1"].mean(),
        "macro_sensitivity": per_class_df["sensitivity"].mean(),
        "macro_specificity": per_class_df["specificity"].mean(),
        "accuracy": accuracy_score(all_labels, y_pred),
    }

    return {"per_class": per_class_df, "macro": macro_metrics}

In [11]:
"""
S1_08_run_all_folds.txt
Script 1 — EfficientNet-B3 (Image-Only Ablation)
Orchestrates all 5 folds. Fold-level resume: skips training for any fold
whose completion marker (fold{n}_training_summary.json, written by
S1_06) already exists -- does NOT rely on best_weights existing, since
that gets saved mid-training too. Evaluation always re-runs per fold
(cheap, idempotent) so the aggregate table is always rebuilt from
current per-fold results.

This is the fixed reference point Phase 6 compares against -- no further
tuning after this completes (per the Runbook's Verify Before Moving On).

Requires already run in this notebook:
  - S1_00_config, S1_06_train_loop, S1_07_metrics
"""

import pandas as pd

N_FOLDS = 5


def run_all_folds():
    macro_results = []

    for fold_num in range(N_FOLDS):
        summary_marker = RESULTS_DIR / f"fold{fold_num}_training_summary.json"

        if summary_marker.exists():
            print(f"Fold {fold_num}: training already complete -- skipping.")
        else:
            train_fold(fold_num)

        eval_result = evaluate_fold(fold_num)
        macro_results.append(eval_result["macro"])

    summary_df = pd.DataFrame(macro_results)
    summary_df.to_csv(RESULTS_DIR / "per_fold_summary.csv", index=False)

    aggregate = summary_df.drop(columns="fold").agg(["mean", "std"])
    aggregate.to_csv(RESULTS_DIR / "aggregate_summary.csv")

    print("\n--- Per-fold summary ---")
    print(summary_df.to_string(index=False))
    print("\n--- Aggregate (mean +/- SD across 5 folds) ---")
    print(aggregate.to_string())

    return summary_df, aggregate

In [12]:
run_all_folds()

C:\Users\primu\AppData\Local\Temp\ipykernel_21336\3366927806.py:72: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:219.)
  weights=torch.as_tensor(sample_weights, dtype=torch.double),


Downloading: "https://download.pytorch.org/models/efficientnet_b3_rwightman-b3899882.pth" to C:\Users\primu/.cache\torch\hub\checkpoints\efficientnet_b3_rwightman-b3899882.pth


100%|██████████| 47.2M/47.2M [00:01<00:00, 45.2MB/s]


Fold 0: starting fresh


Fold 0 | Epoch 0 | train_loss=0.8188 | val_auc=0.7662 | best=0.7662 | patience=0


Fold 0 | Epoch 1 | train_loss=0.3525 | val_auc=0.7592 | best=0.7662 | patience=1


... (146 outputs left out)

Fold 4 | Epoch 25 | train_loss=0.0374 | val_auc=0.9380 | best=0.9511 | patience=8


Fold 4 | Epoch 26 | train_loss=0.0311 | val_auc=0.9443 | best=0.9511 | patience=9


Fold 4 | Epoch 27 | train_loss=0.0293 | val_auc=0.9444 | best=0.9511 | patience=10
Fold 4: early stopping at epoch 27

--- Per-fold summary ---
 fold  macro_auc_roc  macro_f1  macro_sensitivity  macro_specificity  accuracy
    0       0.960045  0.678226           0.756517           0.959569  0.779849
    1       0.947226  0.633494           0.712935           0.951215  0.715618
    2       0.941000  0.615413           0.662983           0.951621  0.755828
    3       0.960059  0.713192           0.714707           0.962280  0.824214
    4       0.951093  0.608317           0.675438           0.954066  0.747086

--- Aggregate (mean +/- SD across 5 folds) ---
      macro_auc_roc  macro_f1  macro_sensitivity  macro_specificity  accuracy
mean       0.951884  0.649728           0.704516           0.955750  0.764519
std        0.008280  0.044710           0.036910           0.004941  0.040526


(   fold  macro_auc_roc  macro_f1  macro_sensitivity  macro_specificity  \
 0     0       0.960045  0.678226           0.756517           0.959569   
 1     1       0.947226  0.633494           0.712935           0.951215   
 2     2       0.941000  0.615413           0.662983           0.951621   
 3     3       0.960059  0.713192           0.714707           0.962280   
 4     4       0.951093  0.608317           0.675438           0.954066   
 
    accuracy  
 0  0.779849  
 1  0.715618  
 2  0.755828  
 3  0.824214  
 4  0.747086  ,
       macro_auc_roc  macro_f1  macro_sensitivity  macro_specificity  accuracy
 mean       0.951884  0.649728           0.704516           0.955750  0.764519
 std        0.008280  0.044710           0.036910           0.004941  0.040526)

# Script 2 [ResNet50]

In [13]:
"""
S2_00_config.txt
Script 2 — ResNet50 (Baseline, No Class Balancing)
Shared configuration — paths, class mapping, hyperparameters, checkpoint settings.
Every other Script 2 file imports from this one.

NOTE: continuing in the same notebook as Script 1. This cell OVERWRITES
Script 1's identically-named config values/functions in memory (OUTPUT_DIR,
checkpoint_path, etc.) -- harmless once Script 1 has finished and its
results are saved to disk, but re-run Script 1's own S1_00 cell first if
you need to go back to it later in this session.
"""

from pathlib import Path

# --- Base paths (same source data as Script 1) ---
BASE_DIR = Path(r"T:\Thesis - Development")
FOLD_ASSIGNMENTS_PATH = BASE_DIR / "Resources" / "Artifacts" / "Phase 1 - P2" / "cv_fold_assignments.csv"
IMAGES_DIR = BASE_DIR / "Resources" / "Artifacts" / "Phase 1 - P2" / "Final Output Phase 1" / "images"

# --- Output paths (this script — ResNet50) ---
OUTPUT_DIR = BASE_DIR / "Resources" / "Artifacts" / "Phase 2 - P1" / "ResNet50"
CHECKPOINTS_DIR = OUTPUT_DIR / "checkpoints"
BEST_WEIGHTS_DIR = OUTPUT_DIR / "best_weights"
RESULTS_DIR = OUTPUT_DIR / "results"

for d in (CHECKPOINTS_DIR, BEST_WEIGHTS_DIR, RESULTS_DIR):
    d.mkdir(parents=True, exist_ok=True)

# --- Fixed class index mapping (same as Script 1 — shared across all Phase 2 scripts) ---
CLASS_TO_IDX = {
    "akiec": 0,
    "bcc": 1,
    "bkl": 2,
    "df": 3,
    "mel": 4,
    "nv": 5,
    "vasc": 6,
}
IDX_TO_CLASS = {v: k for k, v in CLASS_TO_IDX.items()}
NUM_CLASSES = len(CLASS_TO_IDX)

# --- Reproducibility ---
RANDOM_SEED = 42

# --- Image ---
IMAGE_SIZE = 300

# --- Optimizer / training (same as Script 1 — Section 8.3's shared training config) ---
WEIGHT_DECAY = 1e-5
LEARNING_RATE = 1e-4  # flat, whole model — no freeze phase for baselines

# --- Early stopping ---
EARLY_STOP_PATIENCE = 10
MAX_EPOCHS = 100  # raised from 50, matches Script 1's updated value
EARLY_STOP_METRIC = "val_auc_roc"

# --- Checkpointing ---
CHECKPOINT_EVERY_N_EPOCHS = 1     # every epoch
CHECKPOINT_KEEP_HISTORY = False   # rolling only — latest overwrites previous


def checkpoint_path(fold_num: int) -> Path:
    """Rolling checkpoint for a given fold — overwritten each epoch. Crash-recovery only."""
    return CHECKPOINTS_DIR / f"fold{fold_num}_latest.pt"


def best_weights_path(fold_num: int) -> Path:
    """Final best-epoch weights for a given fold — the artifact Phase 6 uses."""
    return BEST_WEIGHTS_DIR / f"fold{fold_num}_best.pt"


print("S2_00_config loaded.")
print(f"  Fold assignments: {FOLD_ASSIGNMENTS_PATH}")
print(f"  Images dir:       {IMAGES_DIR}")
print(f"  Output dir:       {OUTPUT_DIR}")

S2_00_config loaded.
  Fold assignments: T:\Thesis - Development\Resources\Artifacts\Phase 1 - P2\cv_fold_assignments.csv
  Images dir:       T:\Thesis - Development\Resources\Artifacts\Phase 1 - P2\Final Output Phase 1\images
  Output dir:       T:\Thesis - Development\Resources\Artifacts\Phase 2 - P1\ResNet50


In [14]:
"""
S2_01_preprocessing.txt
Script 2 — ResNet50 (Baseline, No Class Balancing)
Image loading + ImageNet normalization. Identical to Script 1's S1_01 —
the underlying data pipeline is shared across all three baselines.
No augmentation here (conditional on split, handled in S2_02_dataset).
"""

import cv2
import numpy as np
import torch

# Requires S2_00_config already run in this notebook (defines IMAGES_DIR).

# Standard ImageNet normalization constants (Phase 1 P2, Section 3.2)
IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)


def load_image(image_id: str) -> np.ndarray:
    """
    Load a processed image by image_id from IMAGES_DIR.
    Returns an RGB uint8 array (H, W, 3). Images are already 300x300 from
    Phase 1 P2's Section 12 pipeline -- no resize applied here.
    """
    img_path = IMAGES_DIR / f"{image_id}.jpg"
    image_bgr = cv2.imread(str(img_path))
    if image_bgr is None:
        raise FileNotFoundError(f"Could not load image: {img_path}")
    return cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)


def normalize_image(image_rgb: np.ndarray) -> torch.Tensor:
    """
    Scale to [0, 1] and apply ImageNet per-channel mean/std normalization.
    Returns a torch.Tensor in C x H x W format.
    """
    image_float = image_rgb.astype(np.float32) / 255.0
    normalized = (image_float - IMAGENET_MEAN) / IMAGENET_STD
    tensor = torch.from_numpy(normalized).permute(2, 0, 1).float()
    return tensor

In [15]:
"""
S2_02_dataset.txt
Script 2 — ResNet50 (Baseline, No Class Balancing)
PyTorch Dataset class + fold dataframe helper. Same structure as Script 1's
S1_02 -- fold assignments are shared across all three models.

Requires already run in this notebook:
  - S2_00_config          (FOLD_ASSIGNMENTS_PATH, CLASS_TO_IDX)
  - S2_01_preprocessing   (load_image, normalize_image)
  - augment_image()       -- already loaded from Script 1's carryover cell,
                              same notebook, no re-paste needed.
"""

import pandas as pd
import torch
from torch.utils.data import Dataset


def get_fold_dataframes(fold_num: int):
    """
    Read cv_fold_assignments.csv and split into (train_df, val_df) for the
    given fold. Training = every row where cv_fold != fold_num (the other
    4 folds combined). Validation = cv_fold == fold_num.
    """
    fold_df = pd.read_csv(FOLD_ASSIGNMENTS_PATH)
    train_df = fold_df[fold_df["cv_fold"] != fold_num].reset_index(drop=True)
    val_df = fold_df[fold_df["cv_fold"] == fold_num].reset_index(drop=True)
    return train_df, val_df


class HAM10000Dataset(Dataset):
    """
    is_train=True  -> augmentation applied (training split)
    is_train=False -> no augmentation (validation split)
    """

    def __init__(self, fold_df: pd.DataFrame, is_train: bool):
        self.fold_df = fold_df.reset_index(drop=True)
        self.is_train = is_train

    def __len__(self):
        return len(self.fold_df)

    def __getitem__(self, idx):
        row = self.fold_df.iloc[idx]
        image_id = row["image_id"]
        label = CLASS_TO_IDX[row["dx"]]

        image_rgb = load_image(image_id)

        if self.is_train:
            image_rgb = augment_image(image_rgb)

        tensor = normalize_image(image_rgb)
        return tensor, torch.tensor(label, dtype=torch.long)

In [16]:
"""
S2_03_model.txt
Script 2 — ResNet50 (Baseline, No Class Balancing)
Model architecture only. No metadata branch, no fusion mechanism, no
freeze/unfreeze phases (baselines train end-to-end from epoch 1).

Pretrained weights: IMAGENET1K_V1 (locked) -- not V2. V2 uses a
substantially different, stronger training recipe (longer schedule,
TrivialAugment, mixup/cutmix, EMA); using it would make ResNet50's
result partly reflect a better pretraining recipe rather than the
architecture itself, undermining Section 8.3's "clean, minimally-modified
reference point" framing and comparability with literature ResNet50
numbers. EfficientNet-B3 has no V2 equivalent in torchvision, so V1
keeps all baselines on equally "standard" pretrained initializations.

Head order: Dropout(0.5) -> Linear(->7) -- same convention as Script 1,
for consistency across all three baselines. ResNet50's default head
(model.fc) is a single Linear layer with no dropout, so this is a
straightforward replacement, not a reordering.

Model outputs RAW LOGITS, not softmax probabilities -- same reasoning
as Script 1 (S1_04): softmax stays inside CrossEntropyLoss during
training; applied explicitly only at inference/metrics time.

Requires already run in this notebook:
  - S2_00_config (NUM_CLASSES)
"""

import torch.nn as nn
from torchvision.models import resnet50, ResNet50_Weights


def build_model(num_classes: int = NUM_CLASSES) -> nn.Module:
    """
    ResNet50, pretrained on ImageNet (V1), with its default 1000-class
    head replaced by a plain classification head for the 7-class task.
    """
    model = resnet50(weights=ResNet50_Weights.IMAGENET1K_V1)

    in_features = model.fc.in_features  # 2048
    model.fc = nn.Sequential(
        nn.Dropout(p=0.5, inplace=True),
        nn.Linear(in_features, num_classes),
    )
    return model

In [17]:
"""
S2_04_checkpointing.txt
Script 2 — ResNet50 (Baseline, No Class Balancing)
Save/load logic for resumable training. Same pattern as Script 1's
S1_05 -- minus the sampler generator state, since Script 2 uses natural
batch sampling (no WeightedRandomSampler).

  - save_checkpoint / load_checkpoint: rolling, per-epoch crash-recovery
    state (checkpoints\). Restores model, optimizer, epoch number, and
    early-stopping state.

  - save_best_weights: separate save (best_weights\) — the final
    per-fold artifact Phase 6 uses. Triggered only on a new best
    validation AUC-ROC.

Requires already run in this notebook:
  - S2_00_config (checkpoint_path, best_weights_path)
"""

import torch


def save_checkpoint(fold_num, epoch, model, optimizer, best_val_auc, patience_counter):
    """
    Overwrites the rolling checkpoint for this fold. Call at the end of
    every completed epoch.
    """
    checkpoint = {
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "best_val_auc": best_val_auc,
        "patience_counter": patience_counter,
    }
    torch.save(checkpoint, checkpoint_path(fold_num))


def load_checkpoint(fold_num, model, optimizer, device="cpu"):
    """
    If a checkpoint exists for this fold: restores model and optimizer
    state IN PLACE, and returns
      {"epoch": ..., "best_val_auc": ..., "patience_counter": ...}
    so the training loop knows where to resume from.

    If no checkpoint exists yet for this fold: returns None (fresh start).
    """
    path = checkpoint_path(fold_num)
    if not path.exists():
        return None

    checkpoint = torch.load(path, map_location=device)
    model.load_state_dict(checkpoint["model_state_dict"])
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])

    return {
        "epoch": checkpoint["epoch"],
        "best_val_auc": checkpoint["best_val_auc"],
        "patience_counter": checkpoint["patience_counter"],
    }


def save_best_weights(fold_num, model):
    """Saves model weights only -- the final artifact for this fold."""
    torch.save(model.state_dict(), best_weights_path(fold_num))

In [18]:
"""
S2_05_train_loop.txt
Script 2 — ResNet50 (Baseline, No Class Balancing)
Per-fold training loop. Same structure as Script 1's S1_06, minus both
balancing mechanisms: standard unweighted CrossEntropyLoss, natural
shuffling instead of a balanced sampler. Resumable at the epoch level
via S2_04's checkpointing.

BATCH_SIZE=16 carried forward as a starting value from Script 1 (same
conservative-until-observed reasoning) -- adjust if a real run shows
different memory headroom for ResNet50.

Requires already run in this notebook:
  - S2_00_config, S2_01_preprocessing, S2_02_dataset, S2_03_model,
    S2_04_checkpointing
"""

import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.utils.data import DataLoader
from sklearn.metrics import roc_auc_score
from tqdm import tqdm

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 16  # starting value -- adjust after a real run shows GPU memory usage


def train_fold(fold_num: int) -> dict:
    """
    Trains ResNet50 (baseline, no class balancing) on the given fold,
    resuming from a checkpoint if one exists. Returns a summary dict
    with the fold's final best_val_auc and how training ended.
    """
    train_df, val_df = get_fold_dataframes(fold_num)

    train_dataset = HAM10000Dataset(train_df, is_train=True)
    val_dataset = HAM10000Dataset(val_df, is_train=False)

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

    model = build_model().to(DEVICE)
    optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    criterion = nn.CrossEntropyLoss()  # unweighted -- no class balancing for this baseline

    # --- Resume if a checkpoint exists for this fold ---
    resumed = load_checkpoint(fold_num, model, optimizer, device=DEVICE)
    if resumed is not None:
        start_epoch = resumed["epoch"] + 1
        best_val_auc = resumed["best_val_auc"]
        patience_counter = resumed["patience_counter"]
        print(f"Fold {fold_num}: resuming from epoch {start_epoch} "
              f"(best_val_auc so far: {best_val_auc:.4f})")
    else:
        start_epoch = 0
        best_val_auc = -float("inf")
        patience_counter = 0
        print(f"Fold {fold_num}: starting fresh")

    early_stopped = False
    final_epoch = start_epoch - 1

    for epoch in range(start_epoch, MAX_EPOCHS):
        # --- Train pass ---
        model.train()
        running_loss = 0.0
        for images, labels in tqdm(train_loader, desc=f"Fold {fold_num} | Epoch {epoch} | train", leave=False):
            images, labels = images.to(DEVICE), labels.to(DEVICE)

            optimizer.zero_grad()
            logits = model(images)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * images.size(0)

        train_loss = running_loss / len(train_dataset)

        # --- Validation pass ---
        model.eval()
        all_labels, all_probs = [], []
        with torch.no_grad():
            for images, labels in val_loader:
                images = images.to(DEVICE)
                logits = model(images)
                probs = torch.softmax(logits, dim=1)
                all_labels.append(labels.numpy())
                all_probs.append(probs.cpu().numpy())

        all_labels = np_concat(all_labels)
        all_probs = np_concat(all_probs)
        val_auc = roc_auc_score(all_labels, all_probs, multi_class="ovr", average="macro")

        # --- Best-checkpoint / early-stopping update ---
        if val_auc > best_val_auc:
            best_val_auc = val_auc
            patience_counter = 0
            save_best_weights(fold_num, model)
        else:
            patience_counter += 1

        final_epoch = epoch
        save_checkpoint(fold_num, epoch, model, optimizer, best_val_auc, patience_counter)

        print(f"Fold {fold_num} | Epoch {epoch} | train_loss={train_loss:.4f} "
              f"| val_auc={val_auc:.4f} | best={best_val_auc:.4f} | patience={patience_counter}")

        if patience_counter >= EARLY_STOP_PATIENCE:
            early_stopped = True
            print(f"Fold {fold_num}: early stopping at epoch {epoch}")
            break

    summary = {
        "fold": fold_num,
        "best_val_auc": best_val_auc,
        "final_epoch": final_epoch,
        "early_stopped": early_stopped,
    }

    # Completion marker -- distinct from best_weights, which gets saved
    # mid-training too. S2_07 checks for THIS file to know a fold is done.
    import json
    with open(RESULTS_DIR / f"fold{fold_num}_training_summary.json", "w") as f:
        json.dump(summary, f)

    return summary


def np_concat(list_of_arrays):
    """Small helper -- avoids importing numpy just for np.concatenate at call sites."""
    import numpy as np
    return np.concatenate(list_of_arrays, axis=0)

In [19]:
"""
S2_06_metrics.txt
Script 2 — ResNet50 (Baseline, No Class Balancing)
Post-training evaluation. Identical structure to Script 1's S1_07. Runs
once per fold, using that fold's saved best weights. Computes BOTH macro
and per-class metrics; sensitivity and specificity are one-vs-rest per class.

Saves two files per fold to RESULTS_DIR:
  - fold{n}_val_predictions.csv  (image_id, true_label, per-class probs)
  - fold{n}_per_class_metrics.csv (class, auc_roc, f1, sensitivity, specificity)

Returns macro metrics + accuracy for S2_07 to aggregate across folds.

Requires already run in this notebook:
  - S2_00_config, S2_02_dataset, S2_03_model, S2_05_train_loop (DEVICE, BATCH_SIZE)
"""

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score, confusion_matrix


def evaluate_fold(fold_num: int) -> dict:
    _, val_df = get_fold_dataframes(fold_num)
    val_dataset = HAM10000Dataset(val_df, is_train=False)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

    model = build_model().to(DEVICE)
    model.load_state_dict(torch.load(best_weights_path(fold_num), map_location=DEVICE))
    model.eval()

    all_labels, all_probs = [], []
    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(DEVICE)
            logits = model(images)
            probs = torch.softmax(logits, dim=1)
            all_labels.append(labels.numpy())
            all_probs.append(probs.cpu().numpy())

    all_labels = np.concatenate(all_labels, axis=0)
    all_probs = np.concatenate(all_probs, axis=0)
    y_pred = all_probs.argmax(axis=1)

    # --- Save raw predictions (image_id, true_label, per-class probs) ---
    pred_df = pd.DataFrame(all_probs, columns=[IDX_TO_CLASS[i] for i in range(NUM_CLASSES)])
    pred_df.insert(0, "true_label", [IDX_TO_CLASS[i] for i in all_labels])
    pred_df.insert(0, "image_id", val_df["image_id"].tolist())
    pred_df.to_csv(RESULTS_DIR / f"fold{fold_num}_val_predictions.csv", index=False)

    # --- Per-class metrics (all one-vs-rest) ---
    cm = confusion_matrix(all_labels, y_pred, labels=range(NUM_CLASSES))
    f1_per_class = f1_score(all_labels, y_pred, labels=list(range(NUM_CLASSES)), average=None, zero_division=0)

    per_class_rows = []
    for i in range(NUM_CLASSES):
        y_true_binary = (all_labels == i).astype(int)
        class_auc = roc_auc_score(y_true_binary, all_probs[:, i])

        tp = cm[i, i]
        fn = cm[i, :].sum() - tp
        fp = cm[:, i].sum() - tp
        tn = cm.sum() - tp - fn - fp

        sensitivity = tp / (tp + fn) if (tp + fn) > 0 else float("nan")
        specificity = tn / (tn + fp) if (tn + fp) > 0 else float("nan")

        per_class_rows.append({
            "class": IDX_TO_CLASS[i],
            "auc_roc": class_auc,
            "f1": f1_per_class[i],
            "sensitivity": sensitivity,
            "specificity": specificity,
        })

    per_class_df = pd.DataFrame(per_class_rows)
    per_class_df.to_csv(RESULTS_DIR / f"fold{fold_num}_per_class_metrics.csv", index=False)

    # --- Macro / aggregate metrics ---
    macro_metrics = {
        "fold": fold_num,
        "macro_auc_roc": per_class_df["auc_roc"].mean(),
        "macro_f1": per_class_df["f1"].mean(),
        "macro_sensitivity": per_class_df["sensitivity"].mean(),
        "macro_specificity": per_class_df["specificity"].mean(),
        "accuracy": accuracy_score(all_labels, y_pred),
    }

    return {"per_class": per_class_df, "macro": macro_metrics}

In [20]:
"""
S2_07_run_all_folds.txt
Script 2 — ResNet50 (Baseline, No Class Balancing)
Orchestrates all 5 folds. Same pattern as Script 1's S1_08. Fold-level
resume: skips training for any fold whose completion marker
(fold{n}_training_summary.json, written by S2_05) already exists --
does NOT rely on best_weights existing, since that gets saved
mid-training too. Evaluation always re-runs per fold (cheap, idempotent).

This is the fixed reference point Phase 6 compares against -- no further
tuning after this completes.

Requires already run in this notebook:
  - S2_00_config, S2_05_train_loop, S2_06_metrics
"""

import pandas as pd

N_FOLDS = 5


def run_all_folds():
    macro_results = []

    for fold_num in range(N_FOLDS):
        summary_marker = RESULTS_DIR / f"fold{fold_num}_training_summary.json"

        if summary_marker.exists():
            print(f"Fold {fold_num}: training already complete -- skipping.")
        else:
            train_fold(fold_num)

        eval_result = evaluate_fold(fold_num)
        macro_results.append(eval_result["macro"])

    summary_df = pd.DataFrame(macro_results)
    summary_df.to_csv(RESULTS_DIR / "per_fold_summary.csv", index=False)

    aggregate = summary_df.drop(columns="fold").agg(["mean", "std"])
    aggregate.to_csv(RESULTS_DIR / "aggregate_summary.csv")

    print("\n--- Per-fold summary ---")
    print(summary_df.to_string(index=False))
    print("\n--- Aggregate (mean +/- SD across 5 folds) ---")
    print(aggregate.to_string())

    return summary_df, aggregate

In [21]:
run_all_folds()

Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to C:\Users\primu/.cache\torch\hub\checkpoints\resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:11<00:00, 8.80MB/s]


Fold 0: starting fresh


Fold 0 | Epoch 0 | train_loss=0.7545 | val_auc=0.9254 | best=0.9254 | patience=0


Fold 0 | Epoch 1 | train_loss=0.6010 | val_auc=0.9465 | best=0.9465 | patience=0


Fold 0 | Epoch 2 | train_loss=0.5293 | val_auc=0.9469 | best=0.9469 | patience=0


... (118 outputs left out)

Fold 4 | Epoch 21 | train_loss=0.2104 | val_auc=0.9494 | best=0.9559 | patience=8


Fold 4 | Epoch 22 | train_loss=0.2096 | val_auc=0.9516 | best=0.9559 | patience=9


Fold 4 | Epoch 23 | train_loss=0.1983 | val_auc=0.9463 | best=0.9559 | patience=10
Fold 4: early stopping at epoch 23

--- Per-fold summary ---
 fold  macro_auc_roc  macro_f1  macro_sensitivity  macro_specificity  accuracy
    0       0.965792  0.637430           0.658945           0.956784  0.835178
    1       0.962614  0.712072           0.683497           0.962050  0.826340
    2       0.942335  0.528399           0.572001           0.949111  0.738345
    3       0.960262  0.651366           0.643838           0.948926  0.815483
    4       0.955918  0.613193           0.638171           0.958400  0.790793

--- Aggregate (mean +/- SD across 5 folds) ---
      macro_auc_roc  macro_f1  macro_sensitivity  macro_specificity  accuracy
mean       0.957384  0.628492           0.639290           0.955054  0.801228
std        0.009151  0.066786           0.041504           0.005831  0.038890


(   fold  macro_auc_roc  macro_f1  macro_sensitivity  macro_specificity  \
 0     0       0.965792  0.637430           0.658945           0.956784   
 1     1       0.962614  0.712072           0.683497           0.962050   
 2     2       0.942335  0.528399           0.572001           0.949111   
 3     3       0.960262  0.651366           0.643838           0.948926   
 4     4       0.955918  0.613193           0.638171           0.958400   
 
    accuracy  
 0  0.835178  
 1  0.826340  
 2  0.738345  
 3  0.815483  
 4  0.790793  ,
       macro_auc_roc  macro_f1  macro_sensitivity  macro_specificity  accuracy
 mean       0.957384  0.628492           0.639290           0.955054  0.801228
 std        0.009151  0.066786           0.041504           0.005831  0.038890)

# Script 3 [ViT-B/16]

In [22]:
"""
S3_00_config.txt
Script 3 — ViT-B/16 (Baseline, No Class Balancing)
Shared configuration — paths, class mapping, hyperparameters, checkpoint settings.
Every other Script 3 file imports from this one.

NOTE: continuing in the same notebook as Scripts 1 and 2. This cell
OVERWRITES their identically-named config values/functions in memory --
harmless once each has finished and its results are saved to disk.

IMAGES_DIR still points to Phase 1 P2's 300x300 cleaned output -- the
resize down to 224x224 (ViT's native resolution) happens in S3_01, not
here. Same source images as Scripts 1 and 2, just resized differently.
"""

from pathlib import Path

# --- Base paths (same source data as Scripts 1 and 2) ---
BASE_DIR = Path(r"T:\Thesis - Development")
FOLD_ASSIGNMENTS_PATH = BASE_DIR / "Resources" / "Artifacts" / "Phase 1 - P2" / "cv_fold_assignments.csv"
IMAGES_DIR = BASE_DIR / "Resources" / "Artifacts" / "Phase 1 - P2" / "Final Output Phase 1" / "images"

# --- Output paths (this script — ViT-B/16) ---
OUTPUT_DIR = BASE_DIR / "Resources" / "Artifacts" / "Phase 2 - P1" / "ViT-B16"
CHECKPOINTS_DIR = OUTPUT_DIR / "checkpoints"
BEST_WEIGHTS_DIR = OUTPUT_DIR / "best_weights"
RESULTS_DIR = OUTPUT_DIR / "results"

for d in (CHECKPOINTS_DIR, BEST_WEIGHTS_DIR, RESULTS_DIR):
    d.mkdir(parents=True, exist_ok=True)

# --- Fixed class index mapping (same as Scripts 1/2 — shared across all Phase 2 scripts) ---
CLASS_TO_IDX = {
    "akiec": 0,
    "bcc": 1,
    "bkl": 2,
    "df": 3,
    "mel": 4,
    "nv": 5,
    "vasc": 6,
}
IDX_TO_CLASS = {v: k for k, v in CLASS_TO_IDX.items()}
NUM_CLASSES = len(CLASS_TO_IDX)

# --- Reproducibility ---
RANDOM_SEED = 42

# --- Image ---
IMAGE_SIZE = 224  # ViT-B/16's native resolution — NOT the 300x300 used by Scripts 1/2

# --- Optimizer / training (same as Scripts 1/2 — Section 8.3's shared training config) ---
WEIGHT_DECAY = 1e-5
LEARNING_RATE = 1e-4  # flat, whole model — no freeze phase for baselines

# --- Early stopping ---
EARLY_STOP_PATIENCE = 10
MAX_EPOCHS = 100
EARLY_STOP_METRIC = "val_auc_roc"

# --- Checkpointing ---
CHECKPOINT_EVERY_N_EPOCHS = 1     # every epoch
CHECKPOINT_KEEP_HISTORY = False   # rolling only — latest overwrites previous


def checkpoint_path(fold_num: int) -> Path:
    """Rolling checkpoint for a given fold — overwritten each epoch. Crash-recovery only."""
    return CHECKPOINTS_DIR / f"fold{fold_num}_latest.pt"


def best_weights_path(fold_num: int) -> Path:
    """Final best-epoch weights for a given fold — the artifact Phase 6 uses."""
    return BEST_WEIGHTS_DIR / f"fold{fold_num}_best.pt"


print("S3_00_config loaded.")
print(f"  Fold assignments: {FOLD_ASSIGNMENTS_PATH}")
print(f"  Images dir:       {IMAGES_DIR}")
print(f"  Output dir:       {OUTPUT_DIR}")
print(f"  Image size:       {IMAGE_SIZE}x{IMAGE_SIZE}")

S3_00_config loaded.
  Fold assignments: T:\Thesis - Development\Resources\Artifacts\Phase 1 - P2\cv_fold_assignments.csv
  Images dir:       T:\Thesis - Development\Resources\Artifacts\Phase 1 - P2\Final Output Phase 1\images
  Output dir:       T:\Thesis - Development\Resources\Artifacts\Phase 2 - P1\ViT-B16
  Image size:       224x224


In [23]:
"""
S3_01_preprocessing.txt
Script 3 — ViT-B/16 (Baseline, No Class Balancing)
Image loading + resize to 224x224 + ImageNet normalization.

Loads the same 300x300 cleaned images as Scripts 1/2, then resizes down
to ViT's native 224x224 using INTER_AREA (matches Phase 1 P2's Section 8
choice for downsizing). No augmentation here -- conditional on split,
handled in S3_02_dataset, and applied BEFORE this resize (augmentation
operates on the full-quality 300x300 image; 224x224 is the final,
ViT-specific step).

Requires already run in this notebook:
  - S3_00_config (IMAGES_DIR, IMAGE_SIZE)
"""

import cv2
import numpy as np
import torch

# Standard ImageNet normalization constants (Phase 1 P2, Section 3.2)
IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)


def load_image(image_id: str) -> np.ndarray:
    """
    Load a processed image by image_id from IMAGES_DIR.
    Returns an RGB uint8 array (H, W, 3) at 300x300 -- the shared
    Phase 1 P2 output, not yet resized for ViT.
    """
    img_path = IMAGES_DIR / f"{image_id}.jpg"
    image_bgr = cv2.imread(str(img_path))
    if image_bgr is None:
        raise FileNotFoundError(f"Could not load image: {img_path}")
    return cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)


def resize_for_vit(image_rgb: np.ndarray) -> np.ndarray:
    """
    Resizes down to IMAGE_SIZE x IMAGE_SIZE (224x224) using INTER_AREA --
    the correct choice for downsizing, matches Phase 1 P2's own resize
    interpolation convention.
    """
    return cv2.resize(image_rgb, (IMAGE_SIZE, IMAGE_SIZE), interpolation=cv2.INTER_AREA)


def normalize_image(image_rgb: np.ndarray) -> torch.Tensor:
    """
    Scale to [0, 1] and apply ImageNet per-channel mean/std normalization.
    Shape-agnostic -- works the same at 224x224 as it did at 300x300.
    Returns a torch.Tensor in C x H x W format.
    """
    image_float = image_rgb.astype(np.float32) / 255.0
    normalized = (image_float - IMAGENET_MEAN) / IMAGENET_STD
    tensor = torch.from_numpy(normalized).permute(2, 0, 1).float()
    return tensor

In [24]:
"""
S3_02_dataset.txt
Script 3 — ViT-B/16 (Baseline, No Class Balancing)
PyTorch Dataset class + fold dataframe helper. Same structure as Scripts
1/2's dataset classes, with one addition: resize_for_vit() runs between
augmentation and normalization (load -> augment if train -> resize to
224x224 -> normalize).

Requires already run in this notebook:
  - S3_00_config          (FOLD_ASSIGNMENTS_PATH, CLASS_TO_IDX)
  - S3_01_preprocessing   (load_image, resize_for_vit, normalize_image)
  - augment_image()       -- already loaded from Script 1's carryover cell,
                              same notebook, no re-paste needed.
"""

import pandas as pd
import torch
from torch.utils.data import Dataset


def get_fold_dataframes(fold_num: int):
    """
    Read cv_fold_assignments.csv and split into (train_df, val_df) for the
    given fold. Training = every row where cv_fold != fold_num (the other
    4 folds combined). Validation = cv_fold == fold_num.
    """
    fold_df = pd.read_csv(FOLD_ASSIGNMENTS_PATH)
    train_df = fold_df[fold_df["cv_fold"] != fold_num].reset_index(drop=True)
    val_df = fold_df[fold_df["cv_fold"] == fold_num].reset_index(drop=True)
    return train_df, val_df


class HAM10000Dataset(Dataset):
    """
    is_train=True  -> augmentation applied (training split), at 300x300
                       before the resize down to 224x224
    is_train=False -> no augmentation (validation split)
    """

    def __init__(self, fold_df: pd.DataFrame, is_train: bool):
        self.fold_df = fold_df.reset_index(drop=True)
        self.is_train = is_train

    def __len__(self):
        return len(self.fold_df)

    def __getitem__(self, idx):
        row = self.fold_df.iloc[idx]
        image_id = row["image_id"]
        label = CLASS_TO_IDX[row["dx"]]

        image_rgb = load_image(image_id)

        if self.is_train:
            image_rgb = augment_image(image_rgb)

        image_rgb = resize_for_vit(image_rgb)
        tensor = normalize_image(image_rgb)
        return tensor, torch.tensor(label, dtype=torch.long)

In [25]:
"""
S3_03_model.txt
Script 3 — ViT-B/16 (Baseline, No Class Balancing)
Model architecture only. No metadata branch, no fusion mechanism, no
freeze/unfreeze phases (baselines train end-to-end from epoch 1).

Pretrained weights: IMAGENET1K_V1 (locked) -- not SWAG. SWAG weights use
a substantially different, non-standard recipe (trained at 384x384,
fine-tuned down; much stronger than a standard ViT-B/16 baseline). Same
reasoning as Script 2's ResNet50 choice: using SWAG would make ViT's
result partly reflect a stronger pretraining recipe rather than the
architecture itself, undermining comparability across the three
baselines and with literature ViT-B/16 numbers.

Head order: Dropout(0.5) -> Linear(->7) -- same convention as Scripts
1/2. ViT's default head (model.heads) is a single Linear layer with no
dropout, so this is a straightforward replacement.

Model outputs RAW LOGITS, not softmax probabilities -- same reasoning
as Scripts 1/2: softmax stays inside CrossEntropyLoss during training;
applied explicitly only at inference/metrics time.

Requires already run in this notebook:
  - S3_00_config (NUM_CLASSES)
"""

import torch.nn as nn
from torchvision.models import vit_b_16, ViT_B_16_Weights


def build_model(num_classes: int = NUM_CLASSES) -> nn.Module:
    """
    ViT-B/16, pretrained on ImageNet (V1), with its default 1000-class
    head replaced by a plain classification head for the 7-class task.
    """
    model = vit_b_16(weights=ViT_B_16_Weights.IMAGENET1K_V1)

    in_features = model.heads.head.in_features  # 768
    model.heads = nn.Sequential(
        nn.Dropout(p=0.5, inplace=True),
        nn.Linear(in_features, num_classes),
    )
    return model

In [26]:
"""
S3_04_checkpointing.txt
Script 3 — ViT-B/16 (Baseline, No Class Balancing)
Save/load logic for resumable training. Identical to Script 2's S2_04 --
no sampler generator state, since this baseline uses natural batch
sampling (no WeightedRandomSampler).

  - save_checkpoint / load_checkpoint: rolling, per-epoch crash-recovery
    state (checkpoints\). Restores model, optimizer, epoch number, and
    early-stopping state.

  - save_best_weights: separate save (best_weights\) — the final
    per-fold artifact Phase 6 uses. Triggered only on a new best
    validation AUC-ROC.

Requires already run in this notebook:
  - S3_00_config (checkpoint_path, best_weights_path)
"""

import torch


def save_checkpoint(fold_num, epoch, model, optimizer, best_val_auc, patience_counter):
    """
    Overwrites the rolling checkpoint for this fold. Call at the end of
    every completed epoch.
    """
    checkpoint = {
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "best_val_auc": best_val_auc,
        "patience_counter": patience_counter,
    }
    torch.save(checkpoint, checkpoint_path(fold_num))


def load_checkpoint(fold_num, model, optimizer, device="cpu"):
    """
    If a checkpoint exists for this fold: restores model and optimizer
    state IN PLACE, and returns
      {"epoch": ..., "best_val_auc": ..., "patience_counter": ...}
    so the training loop knows where to resume from.

    If no checkpoint exists yet for this fold: returns None (fresh start).
    """
    path = checkpoint_path(fold_num)
    if not path.exists():
        return None

    checkpoint = torch.load(path, map_location=device)
    model.load_state_dict(checkpoint["model_state_dict"])
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])

    return {
        "epoch": checkpoint["epoch"],
        "best_val_auc": checkpoint["best_val_auc"],
        "patience_counter": checkpoint["patience_counter"],
    }


def save_best_weights(fold_num, model):
    """Saves model weights only -- the final artifact for this fold."""
    torch.save(model.state_dict(), best_weights_path(fold_num))

In [27]:
"""
S3_05_train_loop.txt
Script 3 — ViT-B/16 (Baseline, No Class Balancing)
Per-fold training loop. Same structure as Script 2's S2_05: standard
unweighted CrossEntropyLoss, natural shuffling (no balanced sampler).
Resumable at the epoch level via S3_04's checkpointing.

BATCH_SIZE=16 carried forward as a starting value -- ViT-B/16's memory
footprint differs from ResNet50/EfficientNet-B3, so this may need
adjusting sooner than the other two once a real run shows GPU usage.

Requires already run in this notebook:
  - S3_00_config, S3_01_preprocessing, S3_02_dataset, S3_03_model,
    S3_04_checkpointing
"""

import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.utils.data import DataLoader
from sklearn.metrics import roc_auc_score
from tqdm import tqdm

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 16  # starting value -- adjust after a real run shows GPU memory usage


def train_fold(fold_num: int) -> dict:
    """
    Trains ViT-B/16 (baseline, no class balancing) on the given fold,
    resuming from a checkpoint if one exists. Returns a summary dict
    with the fold's final best_val_auc and how training ended.
    """
    train_df, val_df = get_fold_dataframes(fold_num)

    train_dataset = HAM10000Dataset(train_df, is_train=True)
    val_dataset = HAM10000Dataset(val_df, is_train=False)

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

    model = build_model().to(DEVICE)
    optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    criterion = nn.CrossEntropyLoss()  # unweighted -- no class balancing for this baseline

    # --- Resume if a checkpoint exists for this fold ---
    resumed = load_checkpoint(fold_num, model, optimizer, device=DEVICE)
    if resumed is not None:
        start_epoch = resumed["epoch"] + 1
        best_val_auc = resumed["best_val_auc"]
        patience_counter = resumed["patience_counter"]
        print(f"Fold {fold_num}: resuming from epoch {start_epoch} "
              f"(best_val_auc so far: {best_val_auc:.4f})")
    else:
        start_epoch = 0
        best_val_auc = -float("inf")
        patience_counter = 0
        print(f"Fold {fold_num}: starting fresh")

    early_stopped = False
    final_epoch = start_epoch - 1

    for epoch in range(start_epoch, MAX_EPOCHS):
        # --- Train pass ---
        model.train()
        running_loss = 0.0
        for images, labels in tqdm(train_loader, desc=f"Fold {fold_num} | Epoch {epoch} | train", leave=False):
            images, labels = images.to(DEVICE), labels.to(DEVICE)

            optimizer.zero_grad()
            logits = model(images)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * images.size(0)

        train_loss = running_loss / len(train_dataset)

        # --- Validation pass ---
        model.eval()
        all_labels, all_probs = [], []
        with torch.no_grad():
            for images, labels in val_loader:
                images = images.to(DEVICE)
                logits = model(images)
                probs = torch.softmax(logits, dim=1)
                all_labels.append(labels.numpy())
                all_probs.append(probs.cpu().numpy())

        all_labels = np_concat(all_labels)
        all_probs = np_concat(all_probs)
        val_auc = roc_auc_score(all_labels, all_probs, multi_class="ovr", average="macro")

        # --- Best-checkpoint / early-stopping update ---
        if val_auc > best_val_auc:
            best_val_auc = val_auc
            patience_counter = 0
            save_best_weights(fold_num, model)
        else:
            patience_counter += 1

        final_epoch = epoch
        save_checkpoint(fold_num, epoch, model, optimizer, best_val_auc, patience_counter)

        print(f"Fold {fold_num} | Epoch {epoch} | train_loss={train_loss:.4f} "
              f"| val_auc={val_auc:.4f} | best={best_val_auc:.4f} | patience={patience_counter}")

        if patience_counter >= EARLY_STOP_PATIENCE:
            early_stopped = True
            print(f"Fold {fold_num}: early stopping at epoch {epoch}")
            break

    summary = {
        "fold": fold_num,
        "best_val_auc": best_val_auc,
        "final_epoch": final_epoch,
        "early_stopped": early_stopped,
    }

    # Completion marker -- distinct from best_weights, which gets saved
    # mid-training too. S3_07 checks for THIS file to know a fold is done.
    import json
    with open(RESULTS_DIR / f"fold{fold_num}_training_summary.json", "w") as f:
        json.dump(summary, f)

    return summary


def np_concat(list_of_arrays):
    """Small helper -- avoids importing numpy just for np.concatenate at call sites."""
    import numpy as np
    return np.concatenate(list_of_arrays, axis=0)

In [28]:
"""
S3_06_metrics.txt
Script 3 — ViT-B/16 (Baseline, No Class Balancing)
Post-training evaluation. Identical structure to Scripts 1/2's metrics
files. Runs once per fold, using that fold's saved best weights.
Computes BOTH macro and per-class metrics; sensitivity and specificity
are one-vs-rest per class.

Saves two files per fold to RESULTS_DIR:
  - fold{n}_val_predictions.csv  (image_id, true_label, per-class probs)
  - fold{n}_per_class_metrics.csv (class, auc_roc, f1, sensitivity, specificity)

Returns macro metrics + accuracy for S3_07 to aggregate across folds.

Requires already run in this notebook:
  - S3_00_config, S3_02_dataset, S3_03_model, S3_05_train_loop (DEVICE, BATCH_SIZE)
"""

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score, confusion_matrix


def evaluate_fold(fold_num: int) -> dict:
    _, val_df = get_fold_dataframes(fold_num)
    val_dataset = HAM10000Dataset(val_df, is_train=False)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

    model = build_model().to(DEVICE)
    model.load_state_dict(torch.load(best_weights_path(fold_num), map_location=DEVICE))
    model.eval()

    all_labels, all_probs = [], []
    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(DEVICE)
            logits = model(images)
            probs = torch.softmax(logits, dim=1)
            all_labels.append(labels.numpy())
            all_probs.append(probs.cpu().numpy())

    all_labels = np.concatenate(all_labels, axis=0)
    all_probs = np.concatenate(all_probs, axis=0)
    y_pred = all_probs.argmax(axis=1)

    # --- Save raw predictions (image_id, true_label, per-class probs) ---
    pred_df = pd.DataFrame(all_probs, columns=[IDX_TO_CLASS[i] for i in range(NUM_CLASSES)])
    pred_df.insert(0, "true_label", [IDX_TO_CLASS[i] for i in all_labels])
    pred_df.insert(0, "image_id", val_df["image_id"].tolist())
    pred_df.to_csv(RESULTS_DIR / f"fold{fold_num}_val_predictions.csv", index=False)

    # --- Per-class metrics (all one-vs-rest) ---
    cm = confusion_matrix(all_labels, y_pred, labels=range(NUM_CLASSES))
    f1_per_class = f1_score(all_labels, y_pred, labels=list(range(NUM_CLASSES)), average=None, zero_division=0)

    per_class_rows = []
    for i in range(NUM_CLASSES):
        y_true_binary = (all_labels == i).astype(int)
        class_auc = roc_auc_score(y_true_binary, all_probs[:, i])

        tp = cm[i, i]
        fn = cm[i, :].sum() - tp
        fp = cm[:, i].sum() - tp
        tn = cm.sum() - tp - fn - fp

        sensitivity = tp / (tp + fn) if (tp + fn) > 0 else float("nan")
        specificity = tn / (tn + fp) if (tn + fp) > 0 else float("nan")

        per_class_rows.append({
            "class": IDX_TO_CLASS[i],
            "auc_roc": class_auc,
            "f1": f1_per_class[i],
            "sensitivity": sensitivity,
            "specificity": specificity,
        })

    per_class_df = pd.DataFrame(per_class_rows)
    per_class_df.to_csv(RESULTS_DIR / f"fold{fold_num}_per_class_metrics.csv", index=False)

    # --- Macro / aggregate metrics ---
    macro_metrics = {
        "fold": fold_num,
        "macro_auc_roc": per_class_df["auc_roc"].mean(),
        "macro_f1": per_class_df["f1"].mean(),
        "macro_sensitivity": per_class_df["sensitivity"].mean(),
        "macro_specificity": per_class_df["specificity"].mean(),
        "accuracy": accuracy_score(all_labels, y_pred),
    }

    return {"per_class": per_class_df, "macro": macro_metrics}

In [29]:
"""
S3_07_run_all_folds.txt
Script 3 — ViT-B/16 (Baseline, No Class Balancing)
Orchestrates all 5 folds. Same pattern as Scripts 1/2's orchestrators.
Fold-level resume: skips training for any fold whose completion marker
(fold{n}_training_summary.json, written by S3_05) already exists --
does NOT rely on best_weights existing, since that gets saved
mid-training too. Evaluation always re-runs per fold (cheap, idempotent).

This is the fixed reference point Phase 6 compares against -- no further
tuning after this completes.

Requires already run in this notebook:
  - S3_00_config, S3_05_train_loop, S3_06_metrics
"""

import pandas as pd

N_FOLDS = 5


def run_all_folds():
    macro_results = []

    for fold_num in range(N_FOLDS):
        summary_marker = RESULTS_DIR / f"fold{fold_num}_training_summary.json"

        if summary_marker.exists():
            print(f"Fold {fold_num}: training already complete -- skipping.")
        else:
            train_fold(fold_num)

        eval_result = evaluate_fold(fold_num)
        macro_results.append(eval_result["macro"])

    summary_df = pd.DataFrame(macro_results)
    summary_df.to_csv(RESULTS_DIR / "per_fold_summary.csv", index=False)

    aggregate = summary_df.drop(columns="fold").agg(["mean", "std"])
    aggregate.to_csv(RESULTS_DIR / "aggregate_summary.csv")

    print("\n--- Per-fold summary ---")
    print(summary_df.to_string(index=False))
    print("\n--- Aggregate (mean +/- SD across 5 folds) ---")
    print(aggregate.to_string())

    return summary_df, aggregate

In [30]:
run_all_folds()

Downloading: "https://download.pytorch.org/models/vit_b_16-c867db91.pth" to C:\Users\primu/.cache\torch\hub\checkpoints\vit_b_16-c867db91.pth


100%|██████████| 330M/330M [00:39<00:00, 8.77MB/s] 


Fold 0: starting fresh


Fold 0 | Epoch 0 | train_loss=0.7689 | val_auc=0.9384 | best=0.9384 | patience=0


Fold 0 | Epoch 1 | train_loss=0.6144 | val_auc=0.9541 | best=0.9541 | patience=0


Fold 0 | Epoch 2 | train_loss=0.5622 | val_auc=0.9512 | best=0.9541 | patience=1


... (112 outputs left out)

Fold 4 | Epoch 20 | train_loss=0.2163 | val_auc=0.9399 | best=0.9548 | patience=8


Fold 4 | Epoch 21 | train_loss=0.2096 | val_auc=0.9329 | best=0.9548 | patience=9


Fold 4 | Epoch 22 | train_loss=0.1875 | val_auc=0.9441 | best=0.9548 | patience=10
Fold 4: early stopping at epoch 22

--- Per-fold summary ---
 fold  macro_auc_roc  macro_f1  macro_sensitivity  macro_specificity  accuracy
    0       0.963946  0.677490           0.642855           0.953311  0.830518
    1       0.959317  0.680262           0.658914           0.955173  0.819930
    2       0.947998  0.604437           0.565402           0.935820  0.788462
    3       0.961227  0.656448           0.655090           0.953182  0.808498
    4       0.954842  0.632544           0.635331           0.951807  0.805944

--- Aggregate (mean +/- SD across 5 folds) ---
      macro_auc_roc  macro_f1  macro_sensitivity  macro_specificity  accuracy
mean       0.957466  0.650236           0.631518           0.949859  0.810670
std        0.006247  0.031990           0.038146           0.007938  0.015811


(   fold  macro_auc_roc  macro_f1  macro_sensitivity  macro_specificity  \
 0     0       0.963946  0.677490           0.642855           0.953311   
 1     1       0.959317  0.680262           0.658914           0.955173   
 2     2       0.947998  0.604437           0.565402           0.935820   
 3     3       0.961227  0.656448           0.655090           0.953182   
 4     4       0.954842  0.632544           0.635331           0.951807   
 
    accuracy  
 0  0.830518  
 1  0.819930  
 2  0.788462  
 3  0.808498  
 4  0.805944  ,
       macro_auc_roc  macro_f1  macro_sensitivity  macro_specificity  accuracy
 mean       0.957466  0.650236           0.631518           0.949859  0.810670
 std        0.006247  0.031990           0.038146           0.007938  0.015811)